In [0]:
#Setup
#Create dataset

import pyspark.sql.functions as F
vol = "/Volumes/workspace/bronze/landing/"

df_trades = (
    spark.read.csv(
        f"{vol}/trades/",
        header=True
    )
    .withColumns(
        {
            "quantity" : F.col("quantity").try_cast("int"),
            "price" : F.col("price").try_cast("decimal(18, 4)"),
            "trade_ts" : F.col("trade_ts").cast("timestamp"),
            "notional" : F.round(
                F.col("quantity").try_cast("int") *
                F.col("price").try_cast("decimal(18, 4)"), 2),
            "signed_qty" : F.when( 
                                  F.col("side") == "BUY", F.col("quantity") )
                            .when(
                                F.col("side") == "SELL", -1 * F.col("quantity")
                            )
#                            ,
#            "_ingest_time" : F.current_timestamp(),
#            "_source_file" : F.col("_metadata.file_path")
        }
    )
)


df_trades.show()

In [0]:
#Task 1
#For prediction, checking disctinct count on trades

print(F'Distinct count {df_trades.select("trade_id").distinct().count()}')

"""
Distinct count - 6280
"""




In [0]:
#Task 1 cont...

from pyspark.sql.window import Window

win = Window.partitionBy("trade_id").orderBy(F.col("trade_ts").desc())

df_trades_ranked = (
    df_trades
    .withColumn("rank", F.row_number().over(win))
)

df_trades_ranked.groupBy("rank").agg(
    F.count("rank").alias("count")
).show()


df_trades_uniq = df_trades_ranked.filter(F.col("rank") == 1).cache()
df_trades_uniq.count()

In [0]:
#Task 2
"""
Prediction- 
    if multiple activity on same trade happened with same trade_ts, then multiple records in same trade_id can have same 1st rank for row_number, rank and dense_rank
    Looks like some trades have multiple activity on same trade_ts
"""

win = Window.partitionBy("trade_id").orderBy(F.col("trade_ts").desc())

df_trades_ranked_01 = (
    df_trades
    .withColumns (
        {
            "row_num" : F.row_number().over(win),
            "rank" : F.rank().over(win),
            "dense_rank" : F.dense_rank().over(win)
        }
    )
)

print(f'row_num count - {df_trades_ranked_01.filter(F.col("row_num") == 1).count()}')
print(f'rank count - {df_trades_ranked_01.filter(F.col("rank") == 1).count()}')
print(f'dense_rank count - {df_trades_ranked_01.filter(F.col("dense_rank") == 1).count()}')

#Checking the trades that have multiple actitivity on same trade_ts
df_trades_same_rank = (
    df_trades_ranked_01.filter(F.col("rank") == 1).groupBy("trade_id").agg(
        F.count("trade_id").alias("count")
        ).filter(F.col("count") > 1)
)

df_trades_same_rank.show()

df_trades.join(df_trades_same_rank, on="trade_id").orderBy(F.col("trade_id")).show()
"""
Looks like the multiple activity on same trade at same trade_ts is pure duplicates.
So it does not matter which we keep.
"""

df_trades.filter(F.col("trade_id") == "T202611020000548").show()

"""
It would have mattered which row_number to pick on same trade_id's differenct activities if they were different activities having different impact on business.

And in that case adding the other field which differentiates the multiple activity on the same trade_id to partitionBy and order by on window function may have produced different results and somehow made row_number deterministic
"""

In [0]:
#Task 3

print("Full data-")
df_trades.groupby("status").agg(
    F.count("status").alias("count")
).orderBy("status").show()


print("1st Row num data-")
df_trades_uniq.groupBy("status").agg(
    F.count("status").alias("count")
).orderBy("status").show()

"""
Answers-
    1. There are fewer NEW in latest because, NEW is the first activity on a trade_id followed by AMEND or CANCEL.
    So when we parition by trade_is and order by trade_ts, we are not keeping the NEW status trades on top and the susequenct activity comes up on top as row number 1.
    And lastest only has number 1
    2. If business user asks for live trades, then will give the count from latest leaving out all the CANCEL.
    because only the NEW and AMEND as latest on any trade_id is the live trade.
"""

print("--------------------------")

In [0]:
#Task 4
"""
Predictions - 
    Yes, it can differ and dropDuplicates does not check if it is dropping the older activity rows on trade_id or not.
"""

df_trades_no_dup = df_trades.dropDuplicates(["trade_id"])

df_trades_no_dup.groupBy("status").agg(
    F.count("*").alias("status_count")
).show()

print( F'Count where status is different : {
(
    df_trades_uniq
    .join(df_trades_no_dup, on = "trade_id", how="inner")
    .filter(
        df_trades_uniq["status"] != df_trades_no_dup["status"]
    )
    .count()
)}')

"""
Drop dplicates count-
+------+------------+
|status|status_count|
+------+------------+
|   NEW|        6123|
|CANCEL|          73|
| AMEND|          84|
+------+------------+


Keep latest count using Window(rowNum)-
+------+-----+
|status|count|
+------+-----+
| AMEND|  184|
|CANCEL|  124|
|   NEW| 5972|
+------+-----+

Live Trade count - 5972+184 = 6156

"""

df_trades_uniq.join(df_trades_no_dup, on="trade_id").filter(
    df_trades_uniq["status"] != df_trades_no_dup["status"]
).select(
    df_trades_uniq["trade_id"],
    df_trades_uniq["status"].alias("latest-status"),
    df_trades_no_dup["status"].alias("no-dup-status")
).show()

"""
DropDuplicates randomly keeps any record from the same trade_id and does not check if it is the latest activity or not.
"""

print("--------------------------")

In [0]:
#Task 5

lag_spec = Window.partitionBy("trade_id").orderBy(F.col("trade_ts"))

df_trades_lag= (
    df_trades
    .dropDuplicates()
    .withColumn("prev_qty", F.lag("quantity", 1).over(lag_spec))
)

df_trades_lag.filter(F.col("prev_qty").isNotNull()).select("trade_id", "trade_ts", "quantity", "prev_qty").show()

In [0]:
#Task 5 b

#df_trades_lag.filter(F.col("status") == "AMEND").show()

print(f'Total Ammend Count - {df_trades_lag.filter(F.col("status")== "AMEND").count()}')
print(f'Changed Ammend Count - \
    {df_trades_lag
    .filter( ( F.col("status")== "AMEND") &
                (F.col("quantity") != F.col("prev_qty")) ).count()}')

"""
Total Ammend Count - 187
Changed Ammend Count -     187
"""

df_trades_top_qty_diff_01 = (
    df_trades_lag
    .filter(F.col("status") == "AMEND")
    .select("trade_id", "status", "trade_ts", "quantity", "prev_qty")
    .withColumn("qty_diff", F.abs(F.col("quantity") - F.col("prev_qty")))
    .orderBy( F.col("qty_diff").desc() ).limit(3)
)


df_trades_top_qty_diff_02 = (
    df_trades_lag
    .filter(F.col("status") == "AMEND")
    .select("trade_id", "status", "trade_ts", "quantity", "prev_qty")
    .withColumn("qty_diff", F.col("quantity") - F.col("prev_qty"))
    .orderBy( F.col("qty_diff").desc() ).limit(3)
)

df_trades_top_qty_diff_01.show()

"""
Task 5 b
Top diff is not legit as prev quantity is negative, which is incorrect. but rest look ok.
+----------------+------+-------------------+--------+--------+--------+
|        trade_id|status|           trade_ts|quantity|prev_qty|qty_diff|
+----------------+------+-------------------+--------+--------+--------+
|T202611020002006| AMEND|2026-11-02 17:00:00|    7226|   -7038|   14264|
|T202611040001386| AMEND|2026-11-04 17:00:00|   13732|    9335|    4397|
|T202611020002184| AMEND|2026-11-02 17:00:00|   13815|    9464|    4351|
+----------------+------+-------------------+--------+--------+--------+
"""


"""
Task 5 c
Lag returns NULL for first row of each partition.

For default, we can use F.coalesce or add 0 as default value in Lag function itself
"""

df_trades_lag_def = (
    df_trades.
    withColumn("prev_qty", F.lag("quantity", 1, 0).over(lag_spec))
)

df_trades_lag_def.select("trade_id", "trade_ts", "quantity", "prev_qty").show(5)


In [0]:
#Task 6 
"""
Prediction -
    with vs without rowsBetween-
    I think both must be same because the as we are partitioning in both cases
    it is going to do cummulative in each group of desk, ordered by trade_ts.
    And even by default without rowBetween, it is going to evaluate cummulatively within that group satrting from first row in that group ordered by trade_ts, and same is the window spec of explicit rowsBetween.
"""

desk_spec = Window.partitionBy(F.col("desk")).orderBy(F.col("trade_ts"))

df_trades_running_01 = (
    df_trades_uniq
    .withColumn(
        "by_desk_01", F.sum("signed_qty").over(
            Window
            .partitionBy(F.col("desk"))
            .orderBy(F.col("trade_ts")) )
        )
    .withColumn(
        "by_desk_02", F.sum("signed_qty").over(
            Window
            .partitionBy(F.col("desk"))
            .orderBy(F.col("trade_ts"))
            .rowsBetween(Window.unboundedPreceding, Window.currentRow) )
        )
    .withColumn(
        "by_desk_ts_01", F.sum("signed_qty").over(
            Window
            .partitionBy("desk", "trade_ts") )
        )
    .withColumn(
        "by_desk_ts_02", F.sum("signed_qty").over(
            Window
            .partitionBy("desk", "trade_ts")
            .orderBy(F.col("trade_ts"), "trade_id")
            .rowsBetween(Window.unboundedPreceding, Window.currentRow) )
        )
    
)


print(f'Count where a and b are different-  {(
    df_trades_running_01
    .select("trade_id", "desk", "trade_ts", "signed_qty", "by_desk_01", "by_desk_02")
    .orderBy("desk", "trade_ts")
    .filter(F.col("by_desk_01") != F.col("by_desk_02"))
    .count() 
)}') 

"""
Looks like when the trade_ts is same, they can get rearranged for calculation and hence the difference
"""


"""
Ques - - What is the default frame when a window has `orderBy`, and when it doesn't?
Ans - when no orderBy, it sends group level calculation to each row in that group, and when order by is present then orders using range between and assigns row level.
Also, when using just order by and there are ties in the orderby column(same trade_ts), it again treats that group as 1 and send total to each row with same(trade_ts), using rowsBetween breaks that time but rowsbetween gets lost in order if no tie breaker added in order by
"""

(
    df_trades_running_01
    .select("desk", "trade_ts", "signed_qty", "by_desk_01", "by_desk_02")
    .groupBy("desk")
    .agg(
        F.sum("signed_qty").alias("tot_signed_qty")
    ).orderBy("desk").show()
)

(
    df_trades_running_01
    .select("desk", "trade_ts", "signed_qty", "by_desk_01", "by_desk_02", "by_desk_ts_01", "by_desk_ts_02")
    .withColumn("desk_rank", F.row_number().over(
        Window.partitionBy("desk").orderBy(F.col("trade_ts").desc()))
    ).filter( (F.col("desk_rank") < 50) & (F.col("desk") == "EQ_CASH") ).orderBy("desk", F.col("desk_rank").desc()).show(50)
    
)



In [0]:
#Task 7

df_trades_desk_percent = (
    df_trades_uniq
    .filter(F.col("status") != "CANCEL" )
    .withColumns( {
        "percent_desk" : 
            ( F.col("notional") / F.sum("notional").over(Window.partitionBy("desk", "currency") ) * 100 ),
        "desk_tot":
            ( F.sum("notional").over(Window.partitionBy("desk","currency")))
        }
    )
)

df_trades_desk_percent.select("desk", "currency", "notional", "desk_tot", "percent_desk").orderBy(F.col("percent_desk").desc()).show(3)

"""
Top 3 percentages of notional for each desk in each currency
+------------+--------+----------+----------+------------+
|        desk|currency|  notional|  desk_tot|percent_desk|
+------------+--------+----------+----------+------------+
|          FX|     XXX|1445339.75|1581024.40|   91.417928|
|     EQ_CASH|     XXX|3580692.70|4544250.73|   78.796108|
|FIXED_INCOME|     XXX|4373835.12|8536570.77|   51.236442|
+------------+--------+----------+----------+------------+
"""


"""
Answers
1. bigger percentage in XXX because they have fewer total rows so division are less meaning 1/10 for XXX vs 1/100 or more for others so percentage becomes smaller for each trade
2. Partition by currency too as trades from 2 currencies can't fall under same unmbrella for calculation
3. Below using Join
    using window function is simpler code wise and may shuffle less unless join uses broadcast, correct me if I am wrong
"""

df_trades_desk_tot_notional = (
    df_trades_uniq
    .filter(F.col("status") != "CANCEL")
    .groupBy("desk", "currency")
    .agg(
        F.sum("notional").alias("tot_notional")
    )
)

#df_trades_desk_tot_notional.show()

df_trades_desk_percent_01 = (
    df_trades_uniq
    .filter(F.col("status") != "CANCEL")
    .join(df_trades_desk_tot_notional, on=["desk", "currency"], how="left")
    .withColumns(
        {
            "percent_desk" :
                ( F.col("notional") / F.col("tot_notional") ) * 100
        }
    )
)


#df_trades_desk_percent_01.filter(F.col("currency") == "XXX").show()

(
    df_trades_desk_percent_01
    .select("desk", "currency", "notional", "tot_notional", "percent_desk")
    .orderBy(F.col("percent_desk").desc())
    .filter(F.col("currency") == "XXX").show(3)
)

print("--------------------")


In [0]:
#Task 8



df_trades_top3_01 = (
    df_trades_uniq
    .withColumn(
        "notional_rank_01",
        F.row_number().over(Window.partitionBy("desk").orderBy(F.col("notional").desc()))
    ).filter(F.col("notional_rank_01") <= 3)
).alias("rn")

df_trades_top3_02 = (
    df_trades_uniq
    .withColumn(
        "notional_rank_01",
        F.dense_rank().over(Window.partitionBy("desk").orderBy(F.col("notional").desc()))
    ).filter(F.col("notional_rank_01") <= 3)
).alias("dr")



(
    df_trades_top3_02
    .select("desk", "trade_id", "notional", "notional_rank_01")
    .show()
)


(
    df_trades_top3_01
    .select("desk", "trade_id", "notional", "notional_rank_01")
    .show()
)
"""
Answers-
-Same rows with both and top match for both dense rank and row number because there is no tie on notional on multple trades.
-Will use row_number if strictly need just the top 3 rows and rank or dense rank if want top on numerical comparison.
    within rank and dense rank, rank if need only 3 records with highest even if match and dense rank if need all top 3 even if multple have ties within top 3

"""

(
    df_trades_top3_02
    .join(df_trades_top3_01, on=["desk", "notional_rank_01"], how="inner")
    .select(
        F.col("rn.desk"), 
        F.col("rn.trade_id"), 
        F.col("dr.trade_id"), 
        F.col("rn.notional").alias("rn_notional"), 
        F.col("dr.notional").alias("dr_notional"), 
        F.col("rn.notional_rank_01").alias("rn"),
        F.col("dr.notional_rank_01").alias("dr"))
    .show()
)




In [0]:
#Task 9

df_trades_tot = (
    df_trades_uniq
    .withColumn(
        "running_qty",
        F.sum(F.col("signed_qty")).over(Window.orderBy(F.col("trade_ts")))
    )
    .withColumn(
        "cnt",
        F.count("*").over(Window.orderBy(F.col("trade_ts")))
    ).agg(
        F.max("running_qty")
    )
).explain()

"""

Running this at scale is dangerous because it will shuffle all the data to a single partition, and the shuffle can be very large

Getting warning -
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
"""

In [0]:
"""
## Say out loud when done (no notes)
1. `groupBy` vs a window function: what's the difference in the output, and when do you use each?
groupBy - 
    gives 1 record per key, groups all the records on the unique value of key; use when need 1 record per group
window - 
    gives the same number of records as input, but we can do cummulative operations like running sum, ranking etc within groups; use when want to do cummulative actions within groups or entire dataset(not recommended on entire set)

2. `row_number` vs `rank` vs `dense_rank`: show the three on the values 100, 90, 90, 80.
row_number - serially assigned number incrementing by 1 per row
    100, 1
    90, 2
    90, 3
    80, 4
rank - assigns based on value but skips the number if there is a tie, and jumps to next rank after tied records
    100, 1
    90, 2
    90, 2
    80, 4
dense_rank - assigns based on value but maintains if there is a tie, and does not jumps to next rank after tied records
    100, 1
    90, 2
    90, 2
    80, 3

3. Why is `dropDuplicates` the wrong tool for "latest row per key"? What do you use instead?
dropDuplicates will keep whichever row it sees first, if combined with orderBy, it can give somewhat expected results

4. What is a window frame? What's the default with `orderBy`, and how can it give a wrong running total?
Window frame is the subset of records within group specified by paritionBy keys,
Default with orderBy is range between unbounded preceding to current row
It can give wrong results because it does not consider if we have ties on the orderBy key, it counts multiple records with same value as 1 record and sends to all rows having key as same value. Better to use rows between and a tie breaker for finer control.

5. What happens physically when you run a window function? Does it shuffle, and on what?
Spark shuffles data having same key value to same partitions and then does sorting within window frame on(eg order by) before running window functions.
Window without partition by send all data to single partitions
Yes, it shuffles on keys specified in partitionBy.




"""
print("---------------THE END-----------------")
